## AND NOW... the Agent Loop!

In [3]:
from agents import Agent, Runner, function_tool, ModelSettings, AsyncOpenAI, OpenAIChatCompletionsModel
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich.console import Console
import docker
import tempfile
import os
# os.environ['LITELLM_LOG'] = 'DEBUG'
import requests
load_dotenv(override=True)

True

In [4]:
todos = []

class ToDoItem(BaseModel):
    description: str = Field(..., description="The text describing the task")
    completed: bool = Field(False, description="Whether the task is complete")

In [5]:
def get_todo_report(print: bool=False) -> str:
    """Get a report of all todos."""
    result = ""
    for index, todo in enumerate(todos):
        completed = "X" if todo.completed else " "
        start = "[strike][green]" if todo.completed else ""
        end = "[/strike][/green]" if todo.completed else ""
        start += "[red]" if "python" in todo.description.lower() else ""
        end += "[/red]" if "python" in todo.description.lower() else ""
        result += f"Todo #{index + 1}: [{completed}] {start}{todo.description}{end}\n"
    if print:
        Console().print(result)
    return result

In [6]:
@function_tool
def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions and return the full list"""
    for desc in descriptions:
        todos.append(ToDoItem(description=desc))
    return get_todo_report(print=True)


@function_tool
def mark_complete(index: int) -> str:
    """Mark complete the todo at the given position (starting from 1) and return the full list"""
    if 1 <= index <= len(todos):
        todos[index - 1].completed = True
    else:
        return "No todo at this index."
    return get_todo_report(print=True)


@function_tool
def list_todos() -> str:
    """Return the full list of todos with completed ones checked off"""
    return get_todo_report()


@function_tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '60 + 80' or '215 / 140'."""
    try:
        # Using a simple eval for basic math (be cautious with production code)
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        return f"Error: {e}"


In [7]:
# 2. Define your settings (this is for the context/hardware)
# LiteLLM passes 'extra' settings through to the provider
settings = ModelSettings(
    tool_choice="auto",
    temperature=0,
    max_completion_tokens=1024,  # Output tokens
    # We pass Ollama-specific options here
)

In [8]:
external_client =  AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM = OpenAIChatCompletionsModel(model='gpt-oss:20b', openai_client=external_client )

In [9]:
instructions = """
You are a Mathematical Auditor that only double checks math. Your primary job is to DOCUMENT the solving of a problem using the To-Do tools.

1. You are NOT allowed to present the final solution in text until EVERY todo item in the list is marked [X].
2. For each turn, you must perform exactly ONE calculation, and then call 'mark_complete' for that specific step.
3. If you have the final answer early, you MUST still step through the 'mark_complete' process for every remaining item before finishing.
4. Your final output must be in Rich console markup (e.g., [bold cyan]4:06:26 PM[/bold cyan]) and must only include the final result.
"""

In [10]:
# instructions = """
# You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
# Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
# Provide your solution in Rich console markup (e.g. [bold red]Error[/bold red]) to indicate colors and styles.
# """
tools = [create_todos, mark_complete, list_todos, calculator]
agent = Agent("Puzzle Agent", model=LLM, instructions=instructions, model_settings=settings, tools=tools)

In [11]:
task = "A train leaves Boston at 2:00 pm traveling 60 mph. Another train leaves New York at 3:00 pm traveling 80 mph toward Boston. When do they meet?"

In [12]:
todos = []
response = await Runner.run(agent, task, max_turns=20)
Console().print("\n\n" + response.final_output)

Todo #1: [ ] Compute distance traveled by first train before second leaves
Todo #2: [ ] Compute remaining distance between trains at 3:00
Todo #3: [ ] Compute relative speed of trains
Todo #4: [ ] Compute time to meet
Todo #5: [ ] Compute meeting time

Todo #1: [X] Compute distance traveled by first train before second leaves
Todo #2: [ ] Compute remaining distance between trains at 3:00
Todo #3: [ ] Compute relative speed of trains
Todo #4: [ ] Compute time to meet
Todo #5: [ ] Compute meeting time

Todo #1: [X] Compute distance traveled by first train before second leaves
Todo #2: [X] Compute remaining distance between trains at 3:00
Todo #3: [ ] Compute relative speed of trains
Todo #4: [ ] Compute time to meet
Todo #5: [ ] Compute meeting time

Todo #1: [X] Compute distance traveled by first train before second leaves
Todo #2: [X] Compute remaining distance between trains at 3:00
Todo #3: [X] Compute relative speed of trains
Todo #4: [ ] Compute time to meet
Todo #5: [ ] Compute meeting time

Todo #1: [X] Compute distance traveled by first train before second leaves
Todo #2: [X] Compute remaining distance between trains at 3:00
Todo #3: [X] Compute relative speed of trains
Todo #4: [X] Compute time to meet
Todo #5: [ ] Compute meeting time

Todo #1: [X] Compute distance traveled by first train before second leaves
Todo #2: [X] Compute remaining distance between trains at 3:00
Todo #3: [X] Compute relative speed of trains
Todo #4: [X] Compute time to meet
Todo #5: [X] Compute meeting time

4:06:26 PM

## AND NOW... the Agent Loop!

In [ ]:
from agents import Agent, Runner, function_tool, ModelSettings
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich.console import Console
import docker
import tempfile
from agents.extensions.models.litellm_model import LitellmModel
import os
os.environ['LITELLM_LOG'] = 'DEBUG'
import requests
load_dotenv(override=True)

True

In [ ]:
import litellm

# Force LiteLLM to treat gemma3 as a tool-calling model
litellm.register_model({
    "ollama/gemma3:12b": {
        "supports_function_calling": True,
        "supports_parallel_function_calling": True,
    }
})

{'ollama/gemma3:12b': {'supports_function_calling': True,
  'supports_parallel_function_calling': True}}

In [ ]:
todos = []

class ToDoItem(BaseModel):
    description: str = Field(..., description="The text describing the task")
    completed: bool = Field(False, description="Whether the task is complete")

In [ ]:
def get_todo_report(print: bool=False) -> str:
    """Get a report of all todos."""
    result = ""
    for index, todo in enumerate(todos):
        completed = "X" if todo.completed else " "
        start = "[strike][green]" if todo.completed else ""
        end = "[/strike][/green]" if todo.completed else ""
        start += "[red]" if "python" in todo.description.lower() else ""
        end += "[/red]" if "python" in todo.description.lower() else ""
        result += f"Todo #{index + 1}: [{completed}] {start}{todo.description}{end}\n"
    if print:
        Console().print(result)
    return result

In [ ]:
@function_tool
def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions and return the full list"""
    for desc in descriptions:
        todos.append(ToDoItem(description=desc))
    return get_todo_report(print=True)


@function_tool
def mark_complete(index: int) -> str:
    """Mark complete the todo at the given position (starting from 1) and return the full list"""
    if 1 <= index <= len(todos):
        todos[index - 1].completed = True
    else:
        return "No todo at this index."
    return get_todo_report(print=True)


@function_tool
def list_todos() -> str:
    """Return the full list of todos with completed ones checked off"""
    return get_todo_report()


In [ ]:
# 2. Define your settings (this is for the context/hardware)
# LiteLLM passes 'extra' settings through to the provider
settings = ModelSettings(
    max_tokens=4096,  # Output tokens
    # We pass Ollama-specific options here
)

In [ ]:
# 3. Initialize the model without the 'num_ctx' in __init__
# Instead, we will pass the context limit when we actually RUN the agent.
LLM = LitellmModel(
    model="ollama/gemma3:12b",
    base_url="http://localhost:11434"
)

In [ ]:
instructions="""
    You only have access to todo list tools. 
    You do NOT have a math tool. 
    Perform all math calculations (like speed and time) yourself within your reasoning 
    before marking a todo as complete. 
    Do not attempt to call any function other than create_todos, mark_complete, or list_todos.
"""
tools = [create_todos, mark_complete, list_todos]
agent = Agent("Puzzle Agent", model=LLM, instructions=instructions, tools=tools)

In [ ]:
# import logging
# import sys

# # Configure logging to output to the console
# logging.basicConfig(
#     level=logging.INFO, # Change to logging.DEBUG for even more detail
#     format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
#     stream=sys.stdout
# )

# # Specifically target the agents library logger if it's too quiet
# logger = logging.getLogger("agents") 
# logger.setLevel(logging.DEBUG)

# # Set LiteLLM to debug mode specifically
# import litellm
# litellm.set_verbose = True 

# logging.getLogger("LiteLLM").setLevel(logging.DEBUG)
import litellm

# This will print the FULL JSON request and response to your console
litellm.set_verbose = True

In [ ]:
task = "Calculate the meeting time step-by-step. 2 PM Train: 60mph. 3 PM Train: 80mph from 210 miles away. Show the math and give the final time in bold."
# Run this without the Todo tools first to see the 3080's speed!

In [ ]:
todos = []
response = await Runner.run(agent, task)
Console().print("\n\n" + response.final_output)

In [ ]:
client = docker.from_env()
image = "python:3.12-slim"

In [ ]:
@function_tool
def execute_python(code: str) -> str:
    """
    Execute the given Python code inside a Docker container with python:3.12-slim,
    and return whatever is printed to stdout.
    You must print the result of the code to stdout in order to retrieve it.
    This uses the python:3.12-slim image and so it does not have scientific libraries installed;
    write simple python 3.12 code using the standard library only. Do not use numpy or scipy.
    IMPORTANT: You must print the result of the code in order to retrieve it.

    Args:
        code: The Python code to run. Remember to print the result.

    """
    print(f"Executing code: {code}")
    with tempfile.TemporaryDirectory() as tmpdir:
        script_path = os.path.join(tmpdir, "script.py")
        volumes = {tmpdir: {"bind": "/tmp", "mode": "ro"}}
        command = ["python", "/tmp/script.py"]
        with open(script_path, "w") as f:
            f.write(code)
        logs = client.containers.run(image=image, command=command, volumes=volumes, remove=True)
    result = logs.decode("utf-8")
    print(f"Result: {result}")
    return result

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def send_push_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

@function_tool
def push(message: str) -> str:
    """Send a text message as a push notification with this brief message

    Args:
        message: The short text message to push
    """

    send_push_notification(message)
    return "Push notification sent"

In [ ]:
instructions = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
You also have access to an execute_python tool to run Python.
To use the execute_python tool, you must have a task on your todo list prefixed with "Write Python code to...".
Write Python code to solve the problem, and then write python to validate your solution to check your work, then use your push tool to send a message with the solution.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution in Rich console markup (e.g. [bold red]Error[/bold red]).
"""
tools = [create_todos, mark_complete, list_todos, execute_python, push]
agent = Agent("Puzzle Agent", model="gpt-4.1-mini", instructions=instructions, tools=tools)

In [ ]:
number = 5 * 11 * 47 * 307
task = f"What are the prime factors of {number}? Reply only with the answer."
todos = []
response = await Runner.run(agent, task)
Console().print("\n\n" + response.final_output)

In [ ]:
client = docker.from_env()
image = "python:3.12-slim"

In [ ]:
@function_tool
def execute_python(code: str) -> str:
    """
    Execute the given Python code inside a Docker container with python:3.12-slim,
    and return whatever is printed to stdout.
    You must print the result of the code to stdout in order to retrieve it.
    This uses the python:3.12-slim image and so it does not have scientific libraries installed;
    write simple python 3.12 code using the standard library only. Do not use numpy or scipy.
    IMPORTANT: You must print the result of the code in order to retrieve it.

    Args:
        code: The Python code to run. Remember to print the result.

    """
    print(f"Executing code: {code}")
    with tempfile.TemporaryDirectory() as tmpdir:
        script_path = os.path.join(tmpdir, "script.py")
        volumes = {tmpdir: {"bind": "/tmp", "mode": "ro"}}
        command = ["python", "/tmp/script.py"]
        with open(script_path, "w") as f:
            f.write(code)
        logs = client.containers.run(image=image, command=command, volumes=volumes, remove=True)
    result = logs.decode("utf-8")
    print(f"Result: {result}")
    return result

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def send_push_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

@function_tool
def push(message: str) -> str:
    """Send a text message as a push notification with this brief message

    Args:
        message: The short text message to push
    """

    send_push_notification(message)
    return "Push notification sent"

In [ ]:
instructions = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
You also have access to an execute_python tool to run Python.
To use the execute_python tool, you must have a task on your todo list prefixed with "Write Python code to...".
Write Python code to solve the problem, and then write python to validate your solution to check your work, then use your push tool to send a message with the solution.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution in Rich console markup (e.g. [bold red]Error[/bold red]).
"""
tools = [create_todos, mark_complete, list_todos, execute_python, push]
agent = Agent("Puzzle Agent", model="gpt-4.1-mini", instructions=instructions, tools=tools)

In [ ]:
number = 5 * 11 * 47 * 307
task = f"What are the prime factors of {number}? Reply only with the answer."
todos = []
response = await Runner.run(agent, task)
Console().print("\n\n" + response.final_output)